# 🏋️ 02 — Train YOLO26m on Sliced Retail Shelf Dataset

This notebook trains the **YOLO26 Medium** model on the sliced dataset
generated by `01_preprocess.ipynb`.

### Hardware Profile
| Component | Spec | Optimization |
|-----------|------|--------------|
| **GPU** | RTX 4060 Ti (16 GB) | AMP (FP16) → ~10 GB VRAM at batch=16 |
| **CPU** | i5-12400F (6 cores) | workers=6 to saturate all cores |
| **RAM** | 32 GB | cache=True for faster data loading |

### Training Recipe
| Setting | Value | Rationale |
|---------|-------|-----------|
| Optimizer | MuSGD | YOLO26's native SGD+Muon hybrid |
| Epochs | 150 | With early stopping (patience=30) |
| Mosaic | 1.0 | Max mosaic for dense scenes |
| MixUp | 0.15 | Moderate blend augmentation |
| Close Mosaic | 15 | Disable mosaic in final 15 epochs |

In [2]:
# ============================================================
#  System & Hardware Check
# ============================================================

import sys
import torch
from pathlib import Path

print(f"Python:       {sys.version}")
print(f"PyTorch:      {torch.__version__}")
print(f"CUDA:         {torch.version.cuda}")
print(f"GPU:          {torch.cuda.get_device_name(0)}")
print(f"cuDNN:        {torch.backends.cudnn.version()}")

assert torch.cuda.is_available(), "CUDA is NOT available! Check your PyTorch installation."
print("\n✅ CUDA is available — ready to train.")

Python:       3.10.20 (main, Jun 11 2026, 15:17:37) [GCC 14.3.0]
PyTorch:      2.12.0+cu132
CUDA:         13.2
GPU:          NVIDIA GeForce RTX 4060 Ti
cuDNN:        92000

✅ CUDA is available — ready to train.


In [3]:
# ============================================================
#  Configuration
# ============================================================

PROJECT_ROOT = Path("..")

# --- Paths ---
DATA_YAML    = str((PROJECT_ROOT / "dataset_sliced" / "data.yaml").resolve())
MODEL_CKPT   = "yolo26m.pt"                      # Ultralytics will auto-download
PROJECT_DIR  = str((PROJECT_ROOT / "runs" / "detect").resolve())
RUN_NAME     = "retail_shelf_yolo26m"

# --- Training Hyperparameters ---
EPOCHS       = 150       # Total epochs (early stopping will cut short if needed)
IMGSZ        = 640       # Matches our tile size
BATCH_SIZE   = 16        # Safe for 16 GB VRAM with AMP
WORKERS      = 6         # Match i5-12400F physical cores
DEVICE       = 0         # GPU index

print(f"Data YAML:  {DATA_YAML}")
print(f"Model:      {MODEL_CKPT}")
print(f"Project:    {PROJECT_DIR}/{RUN_NAME}")
print(f"Batch Size: {BATCH_SIZE}")
print(f"Image Size: {IMGSZ}")
print(f"Workers:    {WORKERS}")

Data YAML:  /home/rakib/Desktop/OFFICE/Projects/ALL Products/dataset_sliced/data.yaml
Model:      yolo26m.pt
Project:    /home/rakib/Desktop/OFFICE/Projects/ALL Products/runs/detect/retail_shelf_yolo26m
Batch Size: 16
Image Size: 640
Workers:    6


## 🔥 Load & Train

In [4]:
# ============================================================
#  Load Pretrained YOLO26 Medium
# ============================================================

from ultralytics import YOLO

model = YOLO(MODEL_CKPT)

print(f"\n✅ Model loaded: {MODEL_CKPT}")
print(f"   Architecture:  YOLO26 Medium")
print(f"   Parameters:    {sum(p.numel() for p in model.model.parameters()) / 1e6:.1f}M")
print(f"   Task:          {model.task}")


✅ Model loaded: yolo26m.pt
   Architecture:  YOLO26 Medium
   Parameters:    21.9M
   Task:          detect


In [5]:
# ============================================================
#  Train — Hardware-Optimized Configuration
# ============================================================
#
#  This cell will run for several hours.
#  Progress is printed live by Ultralytics.
#
#  VRAM estimate: ~10-12 GB with AMP at batch=16
#  If you get OOM, reduce BATCH_SIZE to 12 or 8.
#
# ============================================================

results = model.train(
    # === Data ===
    data       = DATA_YAML,
    epochs     = EPOCHS,
    imgsz      = IMGSZ,
    batch      = BATCH_SIZE,
    
    # === Hardware Optimization ===
    device     = DEVICE,
    amp        = True,           # FP16 mixed precision → halves VRAM usage
    workers    = WORKERS,        # Saturate i5-12400F's 6 cores
    cache      = True,           # RAM-cache dataset (fits in 32 GB)
    
    # === Optimizer (YOLO26 Native) ===
    optimizer  = "MuSGD",        # SGD + Muon hybrid for stable convergence
    
    # === Augmentations (Heavy — prevent overfitting on 600 images) ===
    mosaic     = 1.0,            # Always apply mosaic (4-image merge)
    mixup      = 0.15,           # 15% probability of blending two images
    copy_paste = 0.1,            # 10% copy-paste augmentation
    hsv_h      = 0.015,          # Hue jitter (±1.5%)
    hsv_s      = 0.7,            # Saturation jitter (±70%)
    hsv_v      = 0.4,            # Brightness jitter (±40%)
    degrees    = 5.0,            # Rotation up to ±5°
    translate  = 0.1,            # Translation up to ±10%
    scale      = 0.5,            # Scale jitter (0.5× to 1.5×)
    fliplr     = 0.5,            # 50% horizontal flip
    flipud     = 0.1,            # 10% vertical flip (retail shelves can vary)
    erasing    = 0.1,            # Random erasing for robustness
    close_mosaic = 15,           # Disable mosaic in final 15 epochs (stabilize)
    
    # === Early Stopping & Checkpoints ===
    patience   = 30,             # Stop if no improvement for 30 epochs
    save_period = 10,            # Checkpoint every 10 epochs
    save       = True,
    
    # === Logging ===
    project    = PROJECT_DIR,
    name       = RUN_NAME,
    exist_ok   = True,
    verbose    = True,
    plots      = True,           # Generate built-in metric plots
)

print("\n" + "=" * 60)
print("  ✅ TRAINING COMPLETE")
print("=" * 60)

/home/rakib/miniconda3/envs/yolo/lib/python3.10/site-packages/requests/__init__.py:92: RequestsDependencyWarning: Unable to find acceptable character detection dependency (chardet or charset_normalizer).
  warnings.warn(


New https://pypi.org/project/ultralytics/8.4.77 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.4.67 🚀 Python-3.10.20 torch-2.12.0+cu132 CUDA:0 (NVIDIA GeForce RTX 4060 Ti, 15947MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=True, cfg=None, classes=None, close_mosaic=15, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.1, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/home/rakib/Desktop/OFFICE/Projects/ALL Products/dataset_sliced/data.yaml, degrees=5.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=150, erasing=0.1, exist_ok=True, fliplr=0.5, flipud=0.1, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.15, mode=train, model=yolo

## 📊 Post-Training Summary

In [6]:
# ============================================================
#  Display training results
# ============================================================

import pandas as pd

run_dir = Path(PROJECT_DIR) / RUN_NAME

# Check weights
best_pt = run_dir / "weights" / "best.pt"
last_pt = run_dir / "weights" / "last.pt"

print(f"Run directory:  {run_dir}")
print(f"Best weights:   {best_pt}  [exists: {best_pt.exists()}]")
print(f"Last weights:   {last_pt}  [exists: {last_pt.exists()}]")

# Load and display results
results_csv = run_dir / "results.csv"
if results_csv.exists():
    df = pd.read_csv(results_csv)
    df.columns = df.columns.str.strip()
    
    print(f"\nTraining completed {len(df)} epochs.")
    print(f"\nFinal metrics:")
    print(f"  mAP50:     {df['metrics/mAP50(B)'].iloc[-1]:.4f}")
    print(f"  mAP50-95:  {df['metrics/mAP50-95(B)'].iloc[-1]:.4f}")
    print(f"  Precision: {df['metrics/precision(B)'].iloc[-1]:.4f}")
    print(f"  Recall:    {df['metrics/recall(B)'].iloc[-1]:.4f}")
    
    print(f"\nBest mAP50:     {df['metrics/mAP50(B)'].max():.4f} (epoch {df['metrics/mAP50(B)'].idxmax() + 1})")
    print(f"Best mAP50-95:  {df['metrics/mAP50-95(B)'].max():.4f} (epoch {df['metrics/mAP50-95(B)'].idxmax() + 1})")
else:
    print("\n⚠️  results.csv not found. Check the run directory.")

print(f"\n👉 Next step: Run 04_plot_curves.ipynb, then 03_inference.ipynb")

Run directory:  /home/rakib/Desktop/OFFICE/Projects/ALL Products/runs/detect/retail_shelf_yolo26m
Best weights:   /home/rakib/Desktop/OFFICE/Projects/ALL Products/runs/detect/retail_shelf_yolo26m/weights/best.pt  [exists: True]
Last weights:   /home/rakib/Desktop/OFFICE/Projects/ALL Products/runs/detect/retail_shelf_yolo26m/weights/last.pt  [exists: True]

Training completed 73 epochs.

Final metrics:
  mAP50:     0.7038
  mAP50-95:  0.4580
  Precision: 0.6900
  Recall:    0.6711

Best mAP50:     0.7120 (epoch 43)
Best mAP50-95:  0.4627 (epoch 43)

👉 Next step: Run 04_plot_curves.ipynb, then 03_inference.ipynb
